# Daily Exposures by neighborhood

In [1]:
import os
import folium, cartopy, mapclassify
import geehydro
import geopandas as gpd
import numpy as np, pandas as pd
import cartopy.crs as ccrs
import ee
import rioxarray as rio, xarray as xr
import geemap
from matplotlib import pyplot as plt
import matplotlib.dates as mdates
from time import sleep

In [2]:
# Authenticate and initialise Earth Engine API
try:
    ee.Initialize(project="precise-413717")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project="precise-413717")

Enter verification code:  4/1Ab_5qlmWhcZO9b0nV59cZy-4iZWg20IyVqyh59GDHOqBbI3TsKWFT08pT7I



Successfully saved authorization token.


In [4]:
######### global variable definition #############
##################################################
# paths to geospatial datasets
path='shapefiles/precise_villages.gpkg'
# path=os.path.join(r'/content/precisehealthgeo', path) if IN_COLAB else path
# analysis dates for PRECISE study
start='2018-11-01'
end='2022-12-31'
# UTM coordinate reference systems for study sites
crs_list=dict(gambia='EPSG:32628', mozambique='EPSG:32736', kenya='EPSG:32737')
# ground sampling distance (pixel size in meters) for visualization only
scale_list=dict(landsat=100, modis=1000, era5=11132)
extract_scale=100 # changed this to use uniform small scale because zonal stats for big era-r pixels doesn't work
# spatial axis for xarray plotting
spatial_dims=dict(lon='x', lat='y')

In [78]:
def generateImageCollection(country, dataset, assign_level):

    ### image processing functions ###
    ##################################

    # load the shapefile to geodataframe
    shapefile={
        'neighborhood': f'{country}_neighborhoods',
        'facility': f'{country}_health_facilities',
        'centroid': f'{country}_centroids',
        'district': f'{country}_districts'
    }
    gdf=gpd.read_file(path, layer=shapefile[assign_level])
    # convert gdf to ee feature collection
    roi=(ee.FeatureCollection(geemap.gdf_to_ee(gdf))\
         .set('country', country))

#     # generate image collection for the study period and apply functions
#     if dataset=='landsat':
#         # masks out clouds
#         def mask_clouds(image):
#             # landsat quality assesment band
#             qaBand=image.select('QA_PIXEL')
#             # bits 5 and 3 are cloud and cloud shadow respectively
#             cloudBitMask=1<<5
#             cloudShadowBitMask=1<<3
#             # both bits should be equal to zero indicating clear consitions
#             mask=(qaBand.bitwiseAnd(cloudBitMask).eq(0)\
#                 .And(qaBand.bitwiseAnd(cloudShadowBitMask).eq(0)))
#             # apply the mask to the optical and thermal bands
#             return (image.updateMask(mask)\
#                     .select('SR_B.*', 'ST_B10')
#                     .copyProperties(image, ["system:time_start"]))

#         # applies landsat scaling factors
#         def scale_image(image):
#             opticalBands=image.select('SR_B.*').multiply(0.0000275).add(-0.2)
#             thermalBand=image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('lst')
#             return (image.addBands(opticalBands, None, True)\
#                     .addBands(thermalBand, None, True))

#         # computes Normalised Difference Vegetation Index
#         def calculate_ndvi(image):
#             ndvi=image.normalizedDifference(['SR_B5', 'SR_B4']).rename('ndvi')
#             return image.addBands(ndvi)

#         collection=(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')\
#                     .filterBounds(roi)
#                     .filterDate(start, end) #('2018-11-01', '2018-12-31') this period illustrates clouds for mozambique
#                     .map(mask_clouds)
#                     .map(scale_image)
#                     .map(calculate_ndvi))

#     elif dataset=='landsat-7':
#         # masks out clouds
#         def mask_clouds(image):
#             # landsat quality assesment band
#             qaBand=image.select('QA_PIXEL')
#             # bits 5 and 3 are cloud and cloud shadow respectively
#             cloudBitMask=1<<5
#             cloudShadowBitMask=1<<3
#             # both bits should be equal to zero indicating clear consitions
#             mask=(qaBand.bitwiseAnd(cloudBitMask).eq(0)\
#                 .And(qaBand.bitwiseAnd(cloudShadowBitMask).eq(0)))
#             # apply the mask to the optical and thermal bands
#             return (image.updateMask(mask)\
#                     .select('SR_B.*', 'ST_B6')
#                     .copyProperties(image, ["system:time_start"]))

#         # applies landsat scaling factors
#         def scale_image(image):
#             opticalBands=image.select('SR_B.*').multiply(0.0000275).add(-0.2)
#             thermalBand=image.select('ST_B6').multiply(0.00341802).add(149.0).subtract(273.15).rename('lst')
#             return (image.addBands(opticalBands, None, True)\
#                     .addBands(thermalBand, None, True))

#         # computes Normalised Difference Vegetation Index
#         def calculate_ndvi(image):
#             ndvi=image.normalizedDifference(['SR_B4', 'SR_B3']).rename('ndvi')
#             return image.addBands(ndvi)

#         collection=(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')\
#                     .filterBounds(roi)
#                     .filterDate(start, end)
#                     .map(mask_clouds)
#                     .map(scale_image)
#                     .map(calculate_ndvi))

    if dataset=='modis':
        # applies modis scaling factors
        def scale_image(image):
            return (image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('lst')
                    .copyProperties(image, ["system:time_start"]))
        
        collection=(ee.ImageCollection('MODIS/061/MYD11A1') # daily aqua lst
                    .filterBounds(roi)
                    .filterDate(start, end)
                    .map(scale_image))

    elif dataset=='era5':
        # applies era5 scaling factors
        def scale_image(image): #get era5 skin temperature aka lst
            return (image.select('skin_temperature').subtract(273.15).rename('lst')
                    .copyProperties(image, ["system:time_start"]))

        collection=(ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                    .filterBounds(roi)
                    .filterDate(start, end)
                    .map(scale_image))

    return collection.select('lst'), roi

def generateDailyTimeSeries(country, dataset, assign_level):
    """
    Function to generate a dataframe with daily mean values by assignement level
    """
        
    images, features=generateImageCollection(country, dataset, assign_level)
    
    # assign_level='neighborhood' if assign_level=='centroid' else assign_level

    # Computes mean value for each neighborhood
    def reduceMean(image):
        f_collection=image.reduceRegions(
            reducer=ee.Reducer.mean(),
            collection=features,
            scale=extract_scale,
            crs=crs_list[country])
        return f_collection.map(lambda f: f.set('exposure_date', image.date().format()))

    # Computes mean value for each health facility
    def extractPoint(image):
        f_collection=image.sampleRegions(
            collection=features,
            scale=extract_scale,
            geometries=True,
            projection=crs_list[country])
        return f_collection.map(lambda f: f.set('exposure_date', image.date().format()))
    

    # generate daily mean for either polygons or points
    exposures=images.map(reduceMean) if features.geometry().type().getInfo()[5:]=='Polygon' else images.map(extractPoint)
    # export to dataframe and set new index
    exposures=(geemap.ee_to_df(exposures.flatten())\
               .rename(columns={'mean': 'lst'}))
    # change exposure month datetime format
    exposures['exposure_date']=pd.to_datetime(exposures['exposure_date'])
    
    return exposures
    
#     for col in exposures.columns:
#         if col not in [f'{assign_level}_code', 'exposure_date', exposure]:
#             exposures=exposures.drop(columns=col)
            
#     return (exposures.set_index([f'{assign_level}_code', 'exposure_date'])\
#             .sort_index())

In [6]:
# we can visualise our image collection on a map just to check
def drawCollection(collection, country):
    # map centres
    getCenter=dict(gambia=[13.443, -15.864], mozambique=[-25.1914, 32.7539], kenya=[-3.9995, 39.3609])
    # color palette
    viz=dict(min=0, max=60, palette='6495ed, 32cd32, fdda0d, 8b4000, ff0000')
    # Use folium to visualize the image collection
    map=folium.Map(location=getCenter[country], zoom_start=8)
    map.addLayer(collection[0], viz)
    map.addLayer(collection[1])
    return map

In [12]:
# country='gambia'
# images=generateImageCollection(country, dataset='modis', assign_level='neighborhood')
# drawCollection(images, country)

In [112]:
country='kenya'
exposures=generateDailyTimeSeries(country, dataset='modis', assign_level='centroid')
exposures

,chu,exposure_date,lst,name,neighborhood_code
0,MWELE KISURUTINI,2018-11-01,33.91,SHIKAADABU,254306
1,KOMBENI,2018-11-03,33.69,BATANI,254001
2,KOMBENI,2018-11-03,34.69,BATANI ASILI,254002
3,MWELE KISURUTINI,2018-11-03,34.17,BENGO MWARENI,254003
4,KOMBENI,2018-11-03,35.89,BOFU,254004
...,...,...,...,...,...
22223,MWELE KISURUTINI,2022-12-29,35.23,MWELE B,254062
22224,MWELE KISURUTINI,2022-12-29,27.99,RABAI POWER,254298
22225,MWELE KISURUTINI,2022-12-29,37.63,SHIKAADABU,254306
22226,MWELE KISURUTINI,2022-12-29,27.99,SIMAKENI A,254070


In [113]:
exposures=exposures.sort_values(by=['exposure_date', 'neighborhood_code'])
exposures

,chu,exposure_date,lst,name,neighborhood_code
0,MWELE KISURUTINI,2018-11-01,33.91,SHIKAADABU,254306
1,KOMBENI,2018-11-03,33.69,BATANI,254001
2,KOMBENI,2018-11-03,34.69,BATANI ASILI,254002
3,MWELE KISURUTINI,2018-11-03,34.17,BENGO MWARENI,254003
4,KOMBENI,2018-11-03,35.89,BOFU,254004
...,...,...,...,...,...
22226,MWELE KISURUTINI,2022-12-29,27.99,SIMAKENI A,254070
22227,MWELE KISURUTINI,2022-12-29,27.99,SIMAKENI B,254071
22221,KAMBE,2022-12-29,40.09,MBUNGONI,254238
22224,MWELE KISURUTINI,2022-12-29,27.99,RABAI POWER,254298


In [114]:
exposures[(exposures['neighborhood_code']==254001) & (exposures['exposure_date'].dt.month==11)]

,chu,exposure_date,lst,name,neighborhood_code
1,KOMBENI,2018-11-03,33.69,BATANI,254001
73,KOMBENI,2018-11-05,34.69,BATANI,254001
145,KOMBENI,2018-11-07,34.55,BATANI,254001
216,KOMBENI,2018-11-08,31.37,BATANI,254001
223,KOMBENI,2018-11-12,35.17,BATANI,254001
297,KOMBENI,2018-11-14,36.59,BATANI,254001
370,KOMBENI,2018-11-16,32.19,BATANI,254001
383,KOMBENI,2018-11-17,31.03,BATANI,254001
491,KOMBENI,2018-11-26,36.05,BATANI,254001
537,KOMBENI,2018-11-28,36.47,BATANI,254001


In [43]:
exposures.iloc[-1, 0]-exposures.iloc[0, 0]

Timedelta('1520 days 00:00:00')

In [115]:
exposures.to_csv(r'outputs/daily_temperature/modis_aqua_kenya_centroids.csv', index=False)